# Part 4 · AgentRegistry: a governed catalog for agents and their tools

**The story for the customer:** the first three parts secured *service* traffic with ambient. This part is about *agents*. An engineering org wants developers to ship agents fast, but only from **approved** building blocks, deployed to whichever runtime the platform team allows, with the tools each agent can call governed centrally. AgentRegistry is that control plane: a catalog of approved MCP tool servers, skills and runtimes; a CLI (`arctl`) to scaffold, build and publish an agent; and a one-line deploy to a connected runtime. Governance rides the same agentgateway data plane you already run.

This part runs on **`mesh1` only** and stands alone. Everything is behind one Keycloak IdP, and the same agentgateway that fronts the platform enforces tool-level access at an ambient waypoint.

<div align="center"><svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 740 280" style="width:100%;max-width:1000px;height:auto" font-family="-apple-system,Segoe UI,Roboto,sans-serif">
  <rect x="0" y="0" width="740" height="280" rx="10" fill="#f8fafc"/>
  <text x="370" y="27" text-anchor="middle" font-size="15" font-weight="700" fill="#0f172a">AgentRegistry: the governed catalog and deploy hub</text>
  <defs><marker id="ra" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#334155"/></marker></defs>

  <!-- developer + arctl -->
  <rect x="16" y="108" width="104" height="60" rx="9" fill="#e2e8f0" stroke="#64748b"/>
  <text x="68" y="132" text-anchor="middle" font-size="12" font-weight="600" fill="#334155">developer</text>
  <text x="68" y="150" text-anchor="middle" font-size="10.5" fill="#475569">arctl CLI</text>
  <text x="192" y="122" text-anchor="middle" font-size="9" fill="#475569">init · build · publish</text>
  <line x1="120" y1="138" x2="266" y2="138" stroke="#334155" stroke-width="1.8" marker-end="url(#ra)"/>

  <!-- registry -->
  <rect x="266" y="60" width="210" height="158" rx="10" fill="#e0e7ff" stroke="#6366f1" stroke-width="2"/>
  <text x="371" y="84" text-anchor="middle" font-size="13" font-weight="700" fill="#312e81">AgentRegistry</text>
  <text x="371" y="99" text-anchor="middle" font-size="9.5" fill="#4338ca">in-cluster control plane</text>
  <rect x="284" y="112" width="174" height="26" rx="6" fill="#eef2ff" stroke="#a5b4fc"/>
  <text x="371" y="129" text-anchor="middle" font-size="10.5" fill="#312e81">approved MCP tool servers</text>
  <rect x="284" y="144" width="174" height="26" rx="6" fill="#eef2ff" stroke="#a5b4fc"/>
  <text x="371" y="161" text-anchor="middle" font-size="10.5" fill="#312e81">skills</text>
  <rect x="284" y="176" width="174" height="26" rx="6" fill="#eef2ff" stroke="#a5b4fc"/>
  <text x="371" y="193" text-anchor="middle" font-size="10.5" fill="#312e81">runtimes</text>
  <text x="540" y="96" text-anchor="middle" font-size="9" fill="#475569">deploy</text>
  <line x1="476" y1="110" x2="576" y2="110" stroke="#334155" stroke-width="1.8" marker-end="url(#ra)"/>
  <line x1="476" y1="168" x2="576" y2="168" stroke="#334155" stroke-width="1.8" marker-end="url(#ra)"/>

  <!-- runtimes -->
  <rect x="576" y="92" width="150" height="36" rx="8" fill="#dcfce7" stroke="#16a34a" stroke-width="1.5"/>
  <text x="651" y="110" text-anchor="middle" font-size="11.5" font-weight="600" fill="#14532d">kagent (mesh1)</text>
  <text x="651" y="123" text-anchor="middle" font-size="9" fill="#166534">local runtime</text>
  <rect x="576" y="150" width="150" height="36" rx="8" fill="#fef3c7" stroke="#d97706" stroke-width="1.5"/>
  <text x="651" y="168" text-anchor="middle" font-size="11.5" font-weight="600" fill="#7c2d12">AWS AgentCore</text>
  <text x="651" y="181" text-anchor="middle" font-size="9" fill="#92400e">Bedrock runtime</text>

  <text x="370" y="252" text-anchor="middle" font-size="11" fill="#64748b">A developer scaffolds an agent from approved catalog tools; the registry publishes it and deploys the same agent to any runtime.</text>
</svg></div>

**What you'll run:** browse the approved catalog, scaffold an agent wired to one approved tool, build and publish it, deploy it to kagent and ask it a question, add a second approved tool and watch it appear, then lock the agent down with an AccessPolicy, turn a REST API into MCP tools with no code, and (optionally, with your own AWS session) deploy the *same* agent to AWS Bedrock AgentCore.

## Open the consoles

The platform is reachable from your laptop over the mesh1 LoadBalancer IP via `sslip.io` hostnames (no `/etc/hosts` edits). `Connect` below prints the exact URLs for this cluster.

- **AgentRegistry UI** — the catalog, runtimes, deployments and Access Policies.
- **Keycloak** — the IdP (realm `agentregistry`); log in as `admin-user` / `password`.
- **Swagger Petstore** — https://petstore.swagger.io, the public REST API we turn into MCP tools.

In [ ]:
# Connect: load the mesh1 platform facts, put arctl on PATH, and log the CLI in to
# the in-cluster AgentRegistry as admin-user (Keycloak group admins -> superuser).
cd "$(git rev-parse --show-toplevel)/istio-ambient-demo-kind/agentregistry"
source scripts/connect.sh
echo
echo "  == consoles for this cluster =="
printf "  %-20s %s\n" "AgentRegistry UI:" "http://${AR_HOST}"
printf "  %-20s %s\n" "Keycloak:"         "http://${KEYCLOAK_HOST}  (admin-user / password)"
printf "  %-20s %s\n" "Swagger Petstore:" "https://petstore.swagger.io"

## Reset — return the cluster to a clean demo state

Run this **before** a fresh run (or after, to hand the clusters back to the other labs). It removes what *this* demo creates — the `agentdemo` agent and its kagent deployment, the deployed MCP tool servers, any AccessPolicy + waypoint label, and the Petstore OpenAPI backend — and leaves the platform (kagent, AgentRegistry, Keycloak) and the approved catalog up, so the demo is a clean read and Parts 1-3 (which use their own namespaces) are untouched.

In [ ]:
cd "$(git rev-parse --show-toplevel)/istio-ambient-demo-kind/agentregistry"
source scripts/connect.sh 2>/dev/null
bash scripts/reset.sh

## 1. Browse the registry: approved tools, skills and runtimes

A developer starts here. The catalog holds only what the platform team has approved: two MCP tool servers, one skill, and the runtimes an agent may deploy to.

In [ ]:
echo "  == approved MCP tool servers =="; arctl get mcpservers
echo; echo "  == approved skills ==";          arctl get skills
echo; echo "  == connected runtimes ==";        arctl get runtimes

## 2. Scaffold the agent, wired to ONE approved tool

`arctl init` scaffolds a working agent project (ADK + Python). We wire it to the approved **`my-mcp`** tool server and bake in the approved **`dice-game`** skill. The two `pyproject` fixes are one-time scaffold hygiene, not demo steps: cap Python below 3.14 (the base image is 3.13), and pin a transitive dependency that upstream yanked from PyPI.

In [ ]:
rm -rf agentdemo
arctl init agent agentdemo --framework adk --language python \
  --model-provider anthropic --model-name claude-haiku-4-5 --mcp my-mcp@latest

# scaffold hygiene (see above): keep uv's resolver on a working set for this base image
sed -i '' 's/^requires-python = ">=3.13"/requires-python = ">=3.13,<3.14"/' agentdemo/pyproject.toml
grep -q '\[tool.uv\]' agentdemo/pyproject.toml || cat >> agentdemo/pyproject.toml <<'EOF'

[tool.uv]
override-dependencies = ["opentelemetry-resourcedetector-gcp==1.13.0"]
EOF

# bake the approved dice-game skill into the agent's system instruction
./scripts/add-skill.sh dice-game
echo; echo "  == the agent's system instruction now carries the dice-game skill =="
grep -A2 'build_instruction' agentdemo/agentdemo/agent.py | head -4

## 3. Build the image and publish the agent

`arctl build --push` builds the agent container and pushes it to the cluster's image registry; `arctl apply` publishes the Agent record to the catalog. (`arctl run ./agentdemo` would chat with it locally first; it's interactive, so run that one in a terminal if you want the local loop.)

In [ ]:
arctl build ./agentdemo --push
echo; echo "  == publish the Agent to the catalog =="
arctl apply -f agentdemo/agent.yaml
echo; arctl get agents

## 4. Deploy onto kagent and ask it a question

One `arctl apply` deploys the MCP tool server, one deploys the agent. The registry binds them to the `kind-kagent` runtime and derives the agent's tool list from the MCP servers on that runtime. The agent deploys **keyless**: a Kyverno policy injects the Anthropic key from a Secret when the Agent is created, so the key is never in a manifest or the pod spec.

> `ask.sh` mints an OIDC token *inside* the cluster (as `admin-user`) and calls the agent's A2A endpoint through kagent, so the ask behaves the same in a terminal and in a cell.

In [ ]:
echo "  == deploy the my-mcp tool server =="
arctl apply -f yaml/deploy-mcp-my-mcp.yaml
until kc -n kagent get mcpserver/my-mcp >/dev/null 2>&1; do sleep 2; done
kc -n kagent wait --for=condition=Ready mcpserver/my-mcp --timeout=180s

echo; echo "  == deploy the agent =="
arctl apply -f yaml/deploy-kagent.yaml
# the registry creates the kagent Deployment asynchronously, a beat after apply
# returns, so wait for it to EXIST before waiting on its rollout.
until kc -n kagent get deploy/agentdemo >/dev/null 2>&1; do sleep 2; done
kc -n kagent rollout status deploy/agentdemo --timeout=180s
echo; kc -n kagent get pods | grep -E 'NAME|agentdemo|my-mcp'

In [ ]:
echo "  == ask the agent which tools it has =="
./scripts/ask.sh "List the exact names of every tool you can call. Output only a comma-separated list, nothing else."

## 5. Add a second approved tool, watch it appear

The platform approves a second server, **`everything-server`** (`sum`, `echo`, `printenv`, `reverse_text`, `to_uppercase`). We add it to the agent's `mcpServers`, redeploy, and re-derive the tool list. The delete-then-apply forces a fresh derive (a plain re-apply would be a no-op, since the new tool lives on the Agent, not the Deployment).

In [ ]:
# add everything-server to the agent's approved tool list
python3 - <<'PY'
p="agentdemo/agent.yaml"; s=open(p).read()
if "everything-server" not in s:
    s=s.replace('''  - kind: MCPServer
    name: my-mcp
    tag: latest
''','''  - kind: MCPServer
    name: my-mcp
    tag: latest
  - kind: MCPServer
    name: everything-server
    tag: latest
''')
    open(p,"w").write(s)
PY
arctl apply -f agentdemo/agent.yaml

echo; echo "  == deploy the everything-server tool server =="
arctl apply -f yaml/deploy-mcp-everything-server.yaml
until kc -n kagent get mcpserver/everything-server >/dev/null 2>&1; do sleep 2; done
kc -n kagent wait --for=condition=Ready mcpserver/everything-server --timeout=180s

echo; echo "  == re-derive the agent's tool list (delete + apply) =="
arctl delete deployment agentdemo >/dev/null 2>&1 || true
# the registry reconcile can race a just-Ready MCP server; retry the apply once
arctl apply -f yaml/deploy-kagent.yaml || { sleep 5; arctl apply -f yaml/deploy-kagent.yaml; }
until kc -n kagent get deploy/agentdemo >/dev/null 2>&1; do sleep 2; done
kc -n kagent rollout status deploy/agentdemo --timeout=180s

In [ ]:
echo "  == tools now (the everything-server tools appear) =="
./scripts/ask.sh "List the exact names of every tool you can call. Output only a comma-separated list, nothing else."

In [ ]:
echo "  == a real task: the agent chains its tools (watch the trace) =="
./scripts/ask.sh "Roll a 13-sided die, add 48291 to the result, then tell me if it is prime."

## 6. Govern the tools with an AccessPolicy

The agent can call everything on `everything-server`, including `printenv`. A platform owner cuts that down to least privilege: an **allowlist** naming only `sum`. It is enforced at an **agentgateway waypoint** in front of the MCP server, on the agent's own certificate identity, and it filters the tool from the list, so a denied tool cannot even be seen, let alone called.

<div align="center"><svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 720 250" style="width:100%;max-width:1000px;height:auto" font-family="-apple-system,Segoe UI,Roboto,sans-serif">
  <rect x="0" y="0" width="720" height="250" rx="10" fill="#f8fafc"/>
  <text x="360" y="27" text-anchor="middle" font-size="15" font-weight="700" fill="#0f172a">Least privilege: an AccessPolicy governs the agent's tools at the waypoint</text>
  <defs>
    <marker id="pg" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#16a34a"/></marker>
    <marker id="pr" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#dc2626"/></marker>
  </defs>

  <!-- agent -->
  <rect x="20" y="98" width="120" height="54" rx="9" fill="#dbeafe" stroke="#60a5fa" stroke-width="1.5"/>
  <text x="80" y="122" text-anchor="middle" font-size="12" font-weight="600" fill="#1e293b">agent</text>
  <text x="80" y="139" text-anchor="middle" font-size="9.5" fill="#475569">(kagent)</text>
  <line x1="140" y1="125" x2="206" y2="125" stroke="#334155" stroke-width="1.6" marker-end="url(#pg)"/>

  <!-- waypoint / AccessPolicy -->
  <rect x="206" y="86" width="180" height="78" rx="10" fill="#e0e7ff" stroke="#6366f1" stroke-width="2"/>
  <text x="296" y="110" text-anchor="middle" font-size="12" font-weight="700" fill="#312e81">agentgateway waypoint</text>
  <text x="296" y="127" text-anchor="middle" font-size="10" fill="#4338ca">kagent AccessPolicy</text>
  <text x="296" y="141" text-anchor="middle" font-size="9.5" fill="#4338ca">action: ALLOW</text>
  <text x="296" y="153" text-anchor="middle" font-size="9.5" fill="#4338ca">tools: [ sum ]</text>

  <!-- tools -->
  <line x1="386" y1="112" x2="500" y2="100" stroke="#16a34a" stroke-width="2" marker-end="url(#pg)"/>
  <rect x="500" y="82" width="200" height="30" rx="7" fill="#dcfce7" stroke="#16a34a"/>
  <text x="600" y="102" text-anchor="middle" font-size="11" fill="#14532d">sum   ✓ allowed</text>
  <line x1="386" y1="140" x2="500" y2="152" stroke="#dc2626" stroke-width="2" stroke-dasharray="5 3" marker-end="url(#pr)"/>
  <rect x="500" y="132" width="200" height="42" rx="7" fill="#fee2e2" stroke="#dc2626"/>
  <text x="600" y="150" text-anchor="middle" font-size="10.5" fill="#7f1d1d">echo · printenv · reverse_text</text>
  <text x="600" y="165" text-anchor="middle" font-size="10.5" fill="#7f1d1d">to_uppercase   ✗ denied</text>

  <text x="360" y="212" text-anchor="middle" font-size="11" fill="#64748b">The allowlist names one tool; every other tool on that server is filtered from the agent's list and cannot be called.</text>
</svg></div>

`accesspolicy-on.sh` applies the AccessPolicy, labels the MCP server for the waypoint (the kmcp translator provisions the waypoint Gateway + route), and restarts the agent so it re-lists through the waypoint.

In [ ]:
./scripts/accesspolicy-on.sh

In [ ]:
echo "  == tools now (everything-server is reduced to sum only) =="
./scripts/ask.sh "List the exact names of every tool you can call. Output only a comma-separated list, nothing else."

In [ ]:
echo "  == the policy is a declarative CR =="
kc -n kagent get accesspolicy allow-sum-only -o yaml | sed -n '1,40p'

Revert the governance so the agent sees the full tool list again (leaves the demo re-runnable):

In [ ]:
./scripts/accesspolicy-off.sh

## 7. Turn a REST API into MCP tools (no code)

Not every tool is a purpose-built MCP server. agentgateway reads a plain **OpenAPI** spec and exposes **every operation as an MCP tool**, deriving each tool's input schema from the operation's parameters and body. We point it at the public Swagger Petstore, and its REST operations become callable MCP tools, with the same governance as any other backend.

<div align="center"><svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 720 240" style="width:100%;max-width:1000px;height:auto" font-family="-apple-system,Segoe UI,Roboto,sans-serif">
  <rect x="0" y="0" width="720" height="240" rx="10" fill="#f8fafc"/>
  <text x="360" y="27" text-anchor="middle" font-size="15" font-weight="700" fill="#0f172a">Turn a REST API into governed MCP tools (OpenAPI → MCP)</text>
  <defs><marker id="oa" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#334155"/></marker></defs>

  <!-- REST API -->
  <rect x="20" y="92" width="120" height="56" rx="9" fill="#e2e8f0" stroke="#64748b"/>
  <text x="80" y="114" text-anchor="middle" font-size="12" font-weight="600" fill="#334155">Petstore</text>
  <text x="80" y="131" text-anchor="middle" font-size="9.5" fill="#475569">REST + OpenAPI</text>
  <text x="180" y="110" text-anchor="middle" font-size="9" fill="#475569">spec</text>
  <line x1="140" y1="120" x2="226" y2="120" stroke="#334155" stroke-width="1.6" marker-end="url(#oa)"/>

  <!-- agentgateway -->
  <rect x="226" y="82" width="200" height="76" rx="10" fill="#dcfce7" stroke="#16a34a" stroke-width="1.8"/>
  <text x="326" y="106" text-anchor="middle" font-size="12" font-weight="700" fill="#14532d">agentgateway</text>
  <text x="326" y="123" text-anchor="middle" font-size="10" fill="#166534">EnterpriseAgentgatewayBackend</text>
  <text x="326" y="137" text-anchor="middle" font-size="9.5" fill="#166534">static.protocol: OpenAPI</text>
  <text x="326" y="150" text-anchor="middle" font-size="9.5" fill="#166534">each REST op → an MCP tool</text>
  <text x="470" y="110" text-anchor="middle" font-size="9" fill="#475569">MCP /mcp</text>
  <line x1="426" y1="120" x2="512" y2="120" stroke="#334155" stroke-width="1.6" marker-end="url(#oa)"/>

  <!-- agent -->
  <rect x="512" y="92" width="188" height="56" rx="9" fill="#dbeafe" stroke="#60a5fa" stroke-width="1.5"/>
  <text x="606" y="114" text-anchor="middle" font-size="12" font-weight="600" fill="#1e293b">agent / MCP client</text>
  <text x="606" y="131" text-anchor="middle" font-size="9.5" fill="#475569">calls getPetById, findByStatus…</text>

  <text x="360" y="202" text-anchor="middle" font-size="11" fill="#64748b">No code: agentgateway reads the OpenAPI spec and exposes each operation as an MCP tool the agent can call, with the same governance.</text>
</svg></div>

In [ ]:
echo "  == 1) load the Petstore's live OpenAPI spec into a ConfigMap =="
curl -s https://petstore3.swagger.io/api/v3/openapi.json \
  | kc -n agentgateway-system create configmap petstore-openapi --from-file=schema=/dev/stdin \
      --dry-run=client -o yaml \
  | kc apply -f -

In [ ]:
echo "  == 2) expose it as MCP with protocol: OpenAPI =="
kc -n agentgateway-system apply -f - <<EOF
apiVersion: enterpriseagentgateway.solo.io/v1alpha1
kind: EnterpriseAgentgatewayBackend
metadata: { name: petstore-api, namespace: agentgateway-system }
spec:
  entMcp:
    sessionRouting: Stateless          # a REST API has no MCP session
    failureMode: FailClosed
    targets:
    - name: petstore
      static:
        host: petstore3.swagger.io      # the public Swagger Petstore API
        port: 443
        protocol: OpenAPI               # parse the schema, one tool per operation
        openAPI:
          schemaRef: { name: petstore-openapi }
        policies:
          tls:
            sni: petstore3.swagger.io   # originate TLS to the public HTTPS API
---
apiVersion: gateway.networking.k8s.io/v1
kind: HTTPRoute
metadata: { name: petstore-mcp, namespace: agentgateway-system }
spec:
  parentRefs: [{ name: ar-ingress }]
  hostnames: ["petstore.${LB}.sslip.io"]
  rules:
  - backendRefs: [{ group: enterpriseagentgateway.solo.io, kind: EnterpriseAgentgatewayBackend, name: petstore-api }]
EOF
sleep 5

In [ ]:
MCP="http://petstore.${LB}.sslip.io/"
echo "  == 3) the REST operations are now MCP tools =="
for i in $(seq 1 15); do
  OUT=$(curl -s -m5 -X POST "$MCP" -H "Content-Type: application/json" -H "Accept: application/json,text/event-stream" \
        -d '{"jsonrpc":"2.0","id":1,"method":"tools/list"}')
  echo "$OUT" | grep -q '"name"' && break; sleep 3
done
echo "$OUT" | sed 's/^data: //' | grep -oE '"name":"[^"]+"' | sed 's/"name":/  - /;s/"//g'

In [ ]:
MCP="http://petstore.${LB}.sslip.io/"
call() { curl -s -m8 -X POST "$MCP" -H "Content-Type: application/json" -H "Accept: application/json,text/event-stream" -d "$1" | sed 's/^data: //'; }
echo "  == 4) call them, real Petstore data over MCP =="
echo "  -- addPet (creates pet 777) --"
call '{"jsonrpc":"2.0","id":2,"method":"tools/call","params":{"name":"addPet","arguments":{"body":{"id":777,"name":"Demo Dog","status":"available","photoUrls":["u"]}}}}' \
  | grep -oE '"structuredContent":\{[^}]*\}' | head -1
echo "  -- getPetById 777 (read it back) --"
call '{"jsonrpc":"2.0","id":3,"method":"tools/call","params":{"name":"getPetById","arguments":{"path":{"petId":777}}}}' \
  | grep -oE '"structuredContent":\{[^}]*\}' | head -1

## 8. Same agent, a second runtime: AWS Bedrock AgentCore (optional)

The point of the registry is that one agent record deploys to whatever runtime the platform allows. The same `agentdemo` we deployed to kagent can deploy to **AWS Bedrock AgentCore** as a second runtime, no rewrite. This step needs **your own AWS session** (Bedrock AgentCore, `us-east-1`) and a git repo AgentCore can clone the source from, so it is not run as part of the automated validation. With those in place it is three commands:

```bash
# in a terminal, from istio-ambient-demo-kind/agentregistry:
export SECRETS_FILE=/path/to/your/secrets    # exports AWS_PROFILE, AWS_REGION, AGENT_GIT_URL
source scripts/aws-login.sh                   # your SSO session
./scripts/04d-connect-aws.sh                  # one-time: cross-account role + ECR + register the aws-agentcore runtime
./scripts/agentcore-deploy.sh                 # per-agent: push image to ECR + source to git, deploy to AgentCore
```

`arctl get runtimes` then shows `aws-agentcore` alongside `kind-kagent`, and `arctl get deployments` shows the same agent running on both. The AgentCore deploy publishes its own Agent record (with the ECR image), so it never disturbs the kagent copy.

## Reset / teardown

Re-run the **Reset** cell near the top to hand the clusters back to the other labs. To remove the whole AgentRegistry platform (kagent, AgentRegistry, Keycloak) from `mesh1`, tear the demo suite down with `./setup.sh teardown` from `istio-ambient-demo-kind`.